In [ ]:
import os, pickle, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.axes as axes
import pprint
from tqdm import tqdm
import multiprocessing
import re
from seaborn import heatmap
from functools import reduce
import seaborn as sns
import pandas as pd
import torch
from transformer_lens import HookedTransformer

In [ ]:
def traverse_directory(root_directory):
    """Traverse directory and collect all .pkl file paths."""
    file_paths = []
    for dirpath, _, filenames in os.walk(root_directory):
        for filename in filenames:
            if filename.endswith(".pkl") and "subgraph" not in filename:  # Adjust based on your file format
                file_paths.append(os.path.join(dirpath, filename))
    return file_paths

def extract_model_and_relation(file_path):
    """Extract model name and relation name from file path."""
    path_parts = file_path.split(os.sep)
    model_name = path_parts[-2]  # Model name is the second last part of the path
    relation_name = os.path.splitext(path_parts[-1])[0]  # File name without extension
    return model_name, relation_name

def read_single_file(file_path):
    """Load data from a single .pkl file and return the model, relation, and data."""
    model_name, relation_name = extract_model_and_relation(file_path)
    with open(file_path, 'rb') as file:
        data_entries = pickle.load(file)
    return model_name, relation_name, data_entries

def read_all_files(file_paths):
    """Reads all .pkl files and organizes data by model and relation sequentially."""
    models_data = {}
    total_files = len(file_paths)
    
    print(f"Total files to process: {total_files}")

    # Sequentially read each file
    for file_path in tqdm(file_paths, total=total_files, desc="Processing files"):
        model_name, relation_name, data_entries = read_single_file(file_path)
        if model_name not in models_data:
            models_data[model_name] = {}
        models_data[model_name][relation_name] = data_entries

    return models_data

In [ ]:
root_directory = "/nfs/datz/olmo_models/new_processed_outputs/"

file_paths = traverse_directory(root_directory)
#main_files = [path for path in file_paths if ("subgraph" not in path and "test" not in path and "all_country_capital" in path)]

root_directory = "/nfs/datz/olmo_models/new_processed_outputs/"

all_files = [file for file in file_paths if "test" not in file]

models_data = read_all_files(all_files)

for model_name, relations in models_data.items():
    print(f"Model: {model_name}")
    for relation_name, data_entries in relations.items():
        print(f"  Relation: {relation_name}, Number of Entries: {len(data_entries)}")
        
sorted_models = sorted(
        models_data.keys(), 
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )


In [ ]:
def load_models(model_name, device='cuda:4', checkpoint=""):
    model = HookedTransformer.from_pretrained_no_processing(
        model_name,
        dtype=torch.bfloat16,
        center_unembed=True,
        center_writing_weights=True,
        #fold_ln=True,
        device=device,
        trust_remote_code=True,
        cache_dir="/mounts/data/proj/olmo_models",
        checkpoint_value=checkpoint,
    )

    return model


# Load the models
model_name = "allenai/OLMo-7B-0424-hf"

model = load_models(model_name, device='cuda:7', checkpoint="main")

In [ ]:
ffn_dict = {}

for model_name in sorted_models:
    relations_data = models_data[model_name]

    print(model_name)
    g_total = 0
    general_ffn_map = np.zeros(32, dtype=float)
    e_total = 0
    entity_ffn_map = np.zeros(32, dtype=float)

    relation_answer_ffn_maps = {}
    relation_answer_with_specific = {}

    for relation_name, entries in relations_data.items():

        relation_answer_heatmap = np.zeros(32, dtype=float)
        relation_answer_total = 0
        answer_specific_heatmaps = {}

        for idx, entry in enumerate(entries):

            # Ensure subj_token_span is not None
            if entry["subj_token_span"] is None:
                entry["subj_token_span"] = np.arange(0, len(entry["subject_tokens"])).tolist()

            # Update general heatmap
            general_ffn_map += np.mean(entry['ffns_contribution_maps'], axis=0) #added [:entry['answer_token_span'][0]] because I dont need the contributions from the answer token itself
            g_total += 1

            # Update entity heatmap for subject tokens
            one_data_map = np.zeros(32, dtype=float)
            for t in entry["subj_token_span"]:
                # if t == 0:
                #     e_total -= 1
                #     continue
                one_data_map += entry['ffns_contribution_maps'][t]#[t - 1]
            entity_ffn_map += one_data_map
            e_total += len(entry["subj_token_span"])

            # Update entity heatmap and relation answer heatmap for answer tokens
            one_data_map = np.zeros(32, dtype=float)
            for t in entry["answer_token_span"]:
                one_data_map += entry['ffns_contribution_maps'][t - 1]
            entity_ffn_map += one_data_map
            relation_answer_heatmap += one_data_map

            e_total += len(entry["answer_token_span"])
            relation_answer_total += len(entry["answer_token_span"])

            # Store answer-specific heatmap
            answer_specific_heatmap = one_data_map / len(entry["answer_token_span"])
            answer_specific_heatmaps[idx] = answer_specific_heatmap

        relation_answer_heatmap /= relation_answer_total
        relation_answer_ffn_maps[f"{relation_name}"] = relation_answer_heatmap
        relation_answer_with_specific[f"{relation_name}"] = answer_specific_heatmaps

    # Normalize heatmaps
    general_ffn_map /= g_total
    entity_ffn_map /= e_total

    # Store heatmaps in the dictionary
    ffn_dict[model_name] = {
        'general_ffn_map': general_ffn_map,
        'entity_ffn_map': entity_ffn_map,
        'relation_answer_ffn_maps': relation_answer_ffn_maps,
        'relation_answer_with_specific': relation_answer_with_specific
    }

In [ ]:
def calculate_consistency(heatmaps_dict, THRESHOLD, relation_name, fact_idx):
    overlap_results = {}
    
    # Retrieve the 'main' model's heatmaps
    main_heatmaps = heatmaps_dict['main']

    # Iterate through each model (excluding 'main') and each heatmap type
    for model_name, model_data in heatmaps_dict.items():
        if model_name == 'main':
            continue

        # Store overlaps for this model
        model_overlaps = {}
        
        for heatmap_type in main_heatmaps.keys():
            if heatmap_type == "relation_answer_ffn_maps":
                # Index by relation_name for relation_answer_heatmaps
                main_heatmap = (main_heatmaps[heatmap_type][relation_name] - main_heatmaps['general_ffn_map'] - main_heatmaps['entity_ffn_map']) > THRESHOLD
                model_heatmap = (model_data[heatmap_type][relation_name] - model_data['general_ffn_map'] - model_data['entity_ffn_map']) > THRESHOLD
                overlap_map = main_heatmap == model_heatmap
                count_map = np.logical_and(main_heatmap, model_heatmap)
            elif heatmap_type == "relation_answer_with_specific":
                # Index by relation_name and fact_idx for relation_answer_with_specific
                main_heatmap = (main_heatmaps[heatmap_type][relation_name][fact_idx]  - main_heatmaps['general_ffn_map'] - main_heatmaps['entity_ffn_map'] - main_heatmaps["relation_answer_ffn_maps"][relation_name]) > THRESHOLD
                model_heatmap = (model_data[heatmap_type][relation_name][fact_idx] - model_data['general_ffn_map'] - model_data['entity_ffn_map'] - model_data["relation_answer_ffn_maps"][relation_name]) > THRESHOLD
                overlap_map = main_heatmap == model_heatmap#(main_heatmap - model_heatmap) 
                count_map = np.logical_and(main_heatmap, model_heatmap)
            elif heatmap_type == "entity_ffn_map":
                main_heatmap = (main_heatmaps[heatmap_type] - main_heatmaps['general_ffn_map']) > THRESHOLD
                model_heatmap = (model_data[heatmap_type] - model_data['general_ffn_map']) > THRESHOLD
                overlap_map = main_heatmap == model_heatmap
                count_map = np.logical_and(main_heatmap, model_heatmap)
                # overlap_map = np.logical_and(main_heatmap, model_heatmap)#(main_heatmap - model_heatmap) 
                #continue
            elif heatmap_type == "general_ffn_map":
            #     # General or entity heatmaps
                main_heatmap = main_heatmaps[heatmap_type] > THRESHOLD
                model_heatmap = model_data[heatmap_type] > THRESHOLD
                overlap_map = main_heatmap == model_heatmap
                count_map = np.logical_and(main_heatmap, model_heatmap)

            # Calculate summary statistics, e.g., proportion of overlapping positions
            proportion_overlap = np.sum(overlap_map) / overlap_map.size
            counts = np.sum(count_map)
            
            # Store the overlap map and proportion
            model_overlaps[heatmap_type] = {
                'overlap_map': overlap_map,
                'consistency': proportion_overlap,
                'counts': counts
            }
        
        # Add this model's overlap results to the main dictionary
        overlap_results[model_name] = model_overlaps
    
    return overlap_results

In [ ]:
def calculate_consistency2(heatmaps_dict, THRESHOLD, relation_name, fact_idx):
    overlap_results = {}
    
    # Retrieve the 'main' model's heatmaps
    main_heatmaps = heatmaps_dict['main']

    # Iterate through each model (excluding 'main') and each heatmap type
    for model_name, model_data in heatmaps_dict.items():
        if model_name == 'main':
            continue

        # Store overlaps for this model
        model_overlaps = {}
        
        for heatmap_type in main_heatmaps.keys():
            if heatmap_type == "relation_answer_ffn_maps":
                # Index by relation_name for relation_answer_heatmaps
                main_heatmap = (main_heatmaps[heatmap_type][relation_name]) > THRESHOLD
                model_heatmap = (model_data[heatmap_type][relation_name]) > THRESHOLD
                overlap_map = main_heatmap == model_heatmap
                count_map = np.logical_and(main_heatmap, model_heatmap)
            elif heatmap_type == "relation_answer_with_specific":
                # Index by relation_name and fact_idx for relation_answer_with_specific
                main_heatmap = (main_heatmaps[heatmap_type][relation_name][fact_idx]) > THRESHOLD
                model_heatmap = (model_data[heatmap_type][relation_name][fact_idx]) > THRESHOLD
                overlap_map = main_heatmap == model_heatmap#(main_heatmap - model_heatmap) 
                count_map = np.logical_and(main_heatmap, model_heatmap)
            elif heatmap_type == "entity_ffn_map":
                main_heatmap = (main_heatmaps[heatmap_type]) > THRESHOLD
                model_heatmap = (model_data[heatmap_type]) > THRESHOLD
                overlap_map = main_heatmap == model_heatmap
                count_map = np.logical_and(main_heatmap, model_heatmap)
                # overlap_map = np.logical_and(main_heatmap, model_heatmap)#(main_heatmap - model_heatmap) 
                #continue
            elif heatmap_type == "general_ffn_map":
            #     # General or entity heatmaps
                main_heatmap = main_heatmaps[heatmap_type] > THRESHOLD
                model_heatmap = model_data[heatmap_type] > THRESHOLD
                overlap_map = main_heatmap == model_heatmap
                count_map = np.logical_and(main_heatmap, model_heatmap)

            # Calculate summary statistics, e.g., proportion of overlapping positions
            proportion_overlap = np.sum(overlap_map) / overlap_map.size
            counts = np.sum(count_map)
            
            # Store the overlap map and proportion
            model_overlaps[heatmap_type] = {
                'overlap_map': overlap_map,
                'consistency': proportion_overlap,
                'counts': counts
            }
        
        # Add this model's overlap results to the main dictionary
        overlap_results[model_name] = model_overlaps
    
    return overlap_results

In [ ]:
import matplotlib.pyplot as plt

def plot_ffn_proportion_overlap(overlap_results, relation_name, fact_idx):
    # Prepare data
    model_names = list(overlap_results.keys())  # Assuming already sorted
    general_proportions = []
    entity_proportions = []
    answer_proportions = []
    answer_specific_proportions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_proportions.append(overlap_results[model_name]['general_ffn_map']['consistency'])
        entity_proportions.append(overlap_results[model_name]['entity_ffn_map']['consistency'])
        answer_proportions.append(overlap_results[model_name]['relation_answer_ffn_maps']['consistency'])
        answer_specific_proportions.append(overlap_results[model_name]['relation_answer_with_specific']['consistency'])

    # Plotting
    plt.figure(figsize=(14, 7))
    plt.plot(model_names, general_proportions, label="General FFN Map", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Entity FFN Map", marker='s', linestyle='--')
    plt.plot(model_names, answer_proportions, label="Relation Answer FFN Map", marker='x', linestyle='-.')
    plt.plot(model_names, answer_specific_proportions, label="Answer Specific FFN Map", marker='^', linestyle='-.')

    plt.xticks(rotation=90)
    plt.xlabel("Model Names")
    plt.ylabel("Consistency")
    plt.title(f"FFN Consistency Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def plot_ffn_count_overlap(overlap_results, relation_name, fact_idx):
    # Prepare data
    model_names = list(overlap_results.keys())  # Assuming already sorted
    general_counts = []
    entity_counts = []
    answer_counts = []
    answer_specific_counts = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_counts.append(overlap_results[model_name]['general_ffn_map']['counts'])
        entity_counts.append(overlap_results[model_name]['entity_ffn_map']['counts'])
        answer_counts.append(overlap_results[model_name]['relation_answer_ffn_maps']['counts'])
        answer_specific_counts.append(overlap_results[model_name]['relation_answer_with_specific']['counts'])

    # Plotting
    plt.figure(figsize=(14, 7))
    plt.plot(model_names, general_counts, label="General FFN Map", marker='o', linestyle='-')
    plt.plot(model_names, entity_counts, label="Entity FFN Map", marker='s', linestyle='--')
    plt.plot(model_names, answer_counts, label="Relation Answer FFN Map", marker='x', linestyle='-.')
    plt.plot(model_names, answer_specific_counts, label="Answer Specific FFN Map", marker='^', linestyle='-.')

    plt.xticks(rotation=90)
    plt.xlabel("Model Names")
    plt.ylabel("Counts")
    plt.title(f"FFN Contribution Counts Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
THRESHOLD = 0.95
relation_name = "country_capital_city"
fact_idx = 0
overlap = calculate_consistency(ffn_dict, THRESHOLD, relation_name, fact_idx)
plot_ffn_proportion_overlap(overlap, relation_name, fact_idx)
plot_ffn_count_overlap(overlap, relation_name, fact_idx)

In [ ]:
def calculate_ffn_consistency(models_data):
    ffn_results = {}

    for model_name, relations_data in models_data.items():
        model_consistency = {
            "general_contrib": np.zeros(32, dtype=bool),
            "entity_contrib": np.zeros(32, dtype=bool),
            "relation_answer_contribs": {},
            "relation_answer_with_specific": {}
        }

        for relation_name, entries in relations_data.items():
            relation_contrib = np.zeros(32, dtype=bool)
            data_idx_groups = {}

            # Group entries by data_idx
            for entry in entries:
                if entry["subj_token_span"] is None:
                    entry["subj_token_span"] = np.arange(0, len(entry["subject_tokens"])).tolist()

                data_idx = entry["data_idx"]
                if data_idx not in data_idx_groups:
                    data_idx_groups[data_idx] = []
                data_idx_groups[data_idx].append(entry)

            # Process each data_idx group
            for data_idx, group_entries in data_idx_groups.items():
                specific_contrib = np.zeros(32, dtype=bool)

                for g_entry in group_entries:
                    answer_tokens = [x - 1 for x in g_entry["answer_token_span"]]
                    subj_tokens = [x - 1 for x in g_entry["subj_token_span"]]

                    # Accumulate FFN contributions
                    specific_contrib |= reduce(np.logical_or, g_entry["ffns_contribution_maps"][answer_tokens])
                    model_consistency["entity_contrib"] |= reduce(np.logical_or, g_entry["ffns_contribution_maps"][subj_tokens + answer_tokens])

                model_consistency["relation_answer_with_specific"].setdefault(relation_name, {})[data_idx] = specific_contrib

            # Relation-level contribution
            relation_contrib = reduce(np.logical_or, [
                np.logical_or.reduce(contrib) for contrib in model_consistency["relation_answer_with_specific"].get(relation_name, {}).values()
            ])

            model_consistency["relation_answer_contribs"][relation_name] = relation_contrib

        # General-level contribution
        model_consistency["general_contrib"] = reduce(
            np.logical_or, [np.logical_or.reduce(entry["ffns_contribution_maps"]) for entry in entries]
        )

        ffn_results[model_name] = model_consistency

    return ffn_results

In [ ]:
def plot_ffn_proportion_consistency(ffn_results, relation_name):
    # Prepare data
    model_names = list(ffn_results.keys())  # Already sorted
    general_proportions = []
    entity_proportions = []
    relation_answer_proportions = []

    for model_name in model_names:
        general_proportions.append(np.sum(ffn_results[model_name]["general_contrib"]))
        entity_proportions.append(np.sum(ffn_results[model_name]["entity_contrib"]))
        relation_answer_proportions.append(np.sum(ffn_results[model_name]["relation_answer_contribs"].get(relation_name, np.zeros(32, dtype=bool))))

    # Plotting
    plt.figure(figsize=(14, 7))
    plt.plot(model_names, general_proportions, label="General Contribution", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Entity Contribution", marker='s', linestyle='--')
    plt.plot(model_names, relation_answer_proportions, label=f"Relation Answer Contribution ({relation_name})", marker='x', linestyle='-.')

    plt.xticks(rotation=90)
    plt.xlabel("Model Names")
    plt.ylabel("Proportion of Active FFNs")
    plt.title(f"FFN Consistency Across Models for Relation: {relation_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def plot_ffn_count_consistency(ffn_results, relation_name):
    # Prepare data
    model_names = list(ffn_results.keys())  # Already sorted
    general_counts = []
    entity_counts = []
    relation_answer_counts = []

    for model_name in model_names:
        general_counts.append(np.sum(ffn_results[model_name]["general_contrib"].astype(int)))
        entity_counts.append(np.sum(ffn_results[model_name]["entity_contrib"].astype(int)))
        relation_answer_counts.append(np.sum(ffn_results[model_name]["relation_answer_contribs"].get(relation_name, np.zeros(32, dtype=bool)).astype(int)))

    # Plotting
    plt.figure(figsize=(14, 7))
    plt.plot(model_names, general_counts, label="General Contribution Count", marker='o', linestyle='-')
    plt.plot(model_names, entity_counts, label="Entity Contribution Count", marker='s', linestyle='--')
    plt.plot(model_names, relation_answer_counts, label=f"Relation Answer Contribution Count ({relation_name})", marker='x', linestyle='-.')

    plt.xticks(rotation=90)
    plt.xlabel("Model Names")
    plt.ylabel("Count of Active FFNs")
    plt.title(f"FFN Contribution Counts Across Models for Relation: {relation_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
heatmaps_dict = {}

for model_name in sorted_models:
    relations_data = models_data[model_name]

    print(model_name)
    #g_total = 0
    general_heatmap = np.zeros(32, dtype=bool)
    #e_total = 0
    entity_heatmap = np.zeros(32, dtype=bool)

    relation_answer_heatmaps = {}
    relation_answer_with_specific = {}

    for relation_name, entries in relations_data.items():

        relation_answer_heatmap = np.zeros(32, dtype=bool)
        relation_answer_total = 0
        answer_specific_heatmaps = {}

        for idx, entry in enumerate(entries):

            # Ensure subj_token_span is not None
            if entry["subj_token_span"] is None:
                entry["subj_token_span"] = np.arange(0, len(entry["subject_tokens"])).tolist()

            # Update general heatmap
            general_heatmap += np.any(entry['ffns_contribution_maps'], axis=0)
            #g_total += 1

            # Update entity heatmap for subject tokens
            one_data_map = np.zeros(32, dtype=bool)
            for t in entry["subj_token_span"]:
                #if t == 0:
                #    e_total -= 1
                #    continue
                one_data_map += entry['ffns_contribution_maps'][t - 1]
            entity_heatmap += one_data_map
            e_total += len(entry["subj_token_span"])

            # Update entity heatmap and relation answer heatmap for answer tokens
            one_data_map = np.zeros(32, dtype=bool)
            for t in entry["answer_token_span"]:
                one_data_map += entry['ffns_contribution_maps'][t - 1]
            entity_heatmap += one_data_map
            relation_answer_heatmap += one_data_map

            #e_total += len(entry["answer_token_span"])
            #relation_answer_total += len(entry["answer_token_span"])

            # Store answer-specific heatmap
            answer_specific_heatmap = one_data_map #/ len(entry["answer_token_span"])
            answer_specific_heatmaps[idx] = answer_specific_heatmap

        #relation_answer_heatmap /= relation_answer_total
        relation_answer_heatmaps[f"{relation_name}"] = relation_answer_heatmap
        relation_answer_with_specific[f"{relation_name}"] = answer_specific_heatmaps

    # Normalize heatmaps
    #general_heatmap /= g_total
    #entity_heatmap /= e_total

    # Store heatmaps in the dictionary
    heatmaps_dict[model_name] = {
        'general_heatmap': general_heatmap,
        'entity_heatmap': entity_heatmap,
        'relation_answer_heatmaps': relation_answer_heatmaps,
        'relation_answer_with_specific': relation_answer_with_specific
    }

In [ ]:
def calculate_overlap_with_main(heatmaps_dict, THRESHOLD, relation_name, fact_idx):
    overlap_results = {}
    
    # Retrieve the 'main' model's heatmaps
    main_heatmaps = heatmaps_dict['main']

    # Iterate through each model (excluding 'main') and each heatmap type
    for model_name, model_data in heatmaps_dict.items():
        if model_name == 'main':
            continue

        # Store overlaps for this model
        model_overlaps = {}
        
        for heatmap_type in main_heatmaps.keys():
            if heatmap_type == "relation_answer_heatmaps":
                # Index by relation_name for relation_answer_heatmaps
                main_heatmap = np.logical_and(main_heatmaps[heatmap_type][relation_name], np.logical_not(np.logical_or(main_heatmaps['general_heatmap'],  main_heatmaps['entity_heatmap'])))
                model_heatmap = np.logical_and(model_data[heatmap_type][relation_name], np.logical_not(np.logical_or(model_data['general_heatmap'], model_data['entity_heatmap'])))
                overlap_map = main_heatmap == model_heatmap
            elif heatmap_type == "relation_answer_with_specific":
                # Index by relation_name and fact_idx for relation_answer_with_specific
                main_heatmap = np.logical_and(main_heatmaps[heatmap_type][relation_name][fact_idx], np.logical_not(np.logical_or(main_heatmaps['general_heatmap'],main_heatmaps['entity_heatmap'])))
                model_heatmap = np.logical_and(model_data[heatmap_type][relation_name][fact_idx], np.logical_not(np.logical_or(model_data['general_heatmap'], model_data['entity_heatmap'])))
                overlap_map = main_heatmap == model_heatmap#(main_heatmap - model_heatmap) 
            elif heatmap_type == "entity_heatmap":
                main_heatmap = np.logical_and(main_heatmaps[heatmap_type], np.logical_not(main_heatmaps['general_heatmap']))
                model_heatmap = np.logical_and(model_data[heatmap_type], np.logical_not(model_data['general_heatmap']))
                overlap_map = main_heatmap == model_heatmap
                # overlap_map = np.logical_and(main_heatmap, model_heatmap)#(main_heatmap - model_heatmap) 
                #continue
            elif heatmap_type == "general_heatmap":
            #     # General or entity heatmaps
                main_heatmap = main_heatmaps[heatmap_type] 
                model_heatmap = model_data[heatmap_type]
                overlap_map = main_heatmap == model_heatmap

            # Calculate summary statistics, e.g., proportion of overlapping positions
            #print(np.sum(overlap_map))
            proportion_overlap = np.sum(overlap_map) / overlap_map.size
            
            # Store the overlap map and proportion
            model_overlaps[heatmap_type] = {
                'overlap_map': overlap_map,
                'consistency': proportion_overlap
            }
        
        # Add this model's overlap results to the main dictionary
        overlap_results[model_name] = model_overlaps
    
    return overlap_results

In [ ]:
import matplotlib.pyplot as plt

def plot_proportion_overlap_multiple(overlap_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        overlap_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_proportions = []
    entity_proportions = []
    answer_all_proportions = []
    answer_proportions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_proportions.append(overlap_results[model_name]['general_heatmap']['consistency'])
        entity_proportions.append(overlap_results[model_name]['entity_heatmap']['consistency'])
        answer_all_proportions.append(overlap_results[model_name]['relation_answer_heatmaps']['consistency'])
        answer_proportions.append(overlap_results[model_name]['relation_answer_with_specific']['consistency'])

    # Plotting
    plt.figure(figsize=(14, 7))
    plt.plot(model_names, general_proportions, label="General Heatmap", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Entity Heatmap", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_proportions, label="Relation Answer Heatmap", marker='x', linestyle='-.')
    plt.plot(model_names, answer_proportions, label="Answer Specific Heatmap", marker='^', linestyle='-.')

    plt.xticks(rotation=90)
    plt.xlabel("Model Steps")
    plt.ylabel("Consistency")
    plt.title(f"Consistency Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    

def plot_count_proportion_overlap_multiple(overlap_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        overlap_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_proportions = []
    entity_proportions = []
    answer_all_proportions = []
    answer_proportions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_proportions.append(overlap_results[model_name]['general_heatmap']['counts'])
        entity_proportions.append(overlap_results[model_name]['entity_heatmap']['counts'])
        answer_all_proportions.append(overlap_results[model_name]['relation_answer_heatmaps']['counts'])
        answer_proportions.append(overlap_results[model_name]['relation_answer_with_specific']['counts'])

    # Plotting
    plt.figure(figsize=(14, 7))
    plt.plot(model_names, general_proportions, label="General Heatmap", marker='o', linestyle='-')
    plt.plot(model_names, entity_proportions, label="Entity Heatmap", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_proportions, label="Relation Answer Heatmap", marker='x', linestyle='-.')
    plt.plot(model_names, answer_proportions, label="Answer Specific Heatmap", marker='^', linestyle='-.')

    plt.xticks(rotation=90)
    plt.xlabel("Model Steps")
    plt.ylabel("Counts")
    plt.title(f"Counts of special Heads Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
THRESHOLD = 0.99
relation_name = "books_written"
fact_idx = 0
overlap = calculate_overlap_with_main(heatmaps_dict, THRESHOLD, relation_name, fact_idx)
plot_proportion_overlap_multiple(overlap, relation_name, fact_idx)
plot_count_proportion_overlap_multiple(overlap, relation_name, fact_idx)

In [ ]:
def calculate_switches(heatmaps_dict, THRESHOLD, relation_name, fact_idx):
    switch_results = {}

    # Iterate through consecutive model pairs
    model_names = list(heatmaps_dict.keys())  # Already sorted
    for i in range(len(model_names) - 1):
        model_name = model_names[i]
        next_model_name = model_names[i + 1]

        # Retrieve heatmaps for current and next models
        current_model_heatmaps = heatmaps_dict[model_name]
        next_model_heatmaps = heatmaps_dict[next_model_name]

        # Store switches for this pair
        model_switches = {}

        for heatmap_type in current_model_heatmaps.keys():
            if heatmap_type == "relation_answer_heatmaps":
                # Index by relation_name for relation_answer_heatmaps
                current_heatmap = (
                    current_model_heatmaps[heatmap_type][relation_name]
                    - current_model_heatmaps["general_heatmap"]
                    - current_model_heatmaps["entity_heatmap"]
                ) > THRESHOLD
                next_heatmap = (
                    next_model_heatmaps[heatmap_type][relation_name]
                    - next_model_heatmaps["general_heatmap"]
                    - next_model_heatmaps["entity_heatmap"]
                ) > THRESHOLD

            elif heatmap_type == "relation_answer_with_specific":
                # Index by relation_name and fact_idx for relation_answer_with_specific
                current_heatmap = (
                    current_model_heatmaps[heatmap_type][relation_name][fact_idx]
                    - current_model_heatmaps["general_heatmap"]
                    - current_model_heatmaps["entity_heatmap"]
                    - current_model_heatmaps["relation_answer_heatmaps"][relation_name]
                ) > THRESHOLD
                next_heatmap = (
                    next_model_heatmaps[heatmap_type][relation_name][fact_idx]
                    - next_model_heatmaps["general_heatmap"]
                    - next_model_heatmaps["entity_heatmap"]
                    - next_model_heatmaps["relation_answer_heatmaps"][relation_name]
                ) > THRESHOLD

            elif heatmap_type == "entity_heatmap":
                current_heatmap = (
                    current_model_heatmaps[heatmap_type]
                    - current_model_heatmaps["general_heatmap"]
                ) > THRESHOLD
                next_heatmap = (
                    next_model_heatmaps[heatmap_type]
                    - next_model_heatmaps["general_heatmap"]
                ) > THRESHOLD

            elif heatmap_type == "general_heatmap":
                current_heatmap = current_model_heatmaps[heatmap_type] > THRESHOLD
                next_heatmap = next_model_heatmaps[heatmap_type] > THRESHOLD

            # Calculate switches (differences) between the two models
            switch_map = current_heatmap != next_heatmap
            switch_count = np.sum(switch_map)

            # Store switch statistics
            model_switches[heatmap_type] = {
                "switch_map": switch_map,
                "switch_count": switch_count,
                "switch_proportion": switch_count / switch_map.size,
            }

        # Add results for this model pair
        switch_results[f"{model_name}"] = model_switches

    return switch_results

In [ ]:
def plot_proportion_switches(switch_results, relation_name, fact_idx):
    # Prepare data
    model_pairs = list(switch_results.keys())  # Already sorted
    general_proportions = []
    entity_proportions = []
    answer_proportions = []
    answer_specific_proportions = []

    # Populate lists for each heatmap type
    for model_pair in model_pairs:
        general_proportions.append(switch_results[model_pair]['general_ffn_map']['switch_proportion'])
        entity_proportions.append(switch_results[model_pair]['entity_ffn_map']['switch_proportion'])
        answer_proportions.append(switch_results[model_pair]['relation_answer_ffn_maps']['switch_proportion'])
        answer_specific_proportions.append(switch_results[model_pair]['relation_answer_with_specific']['switch_proportion'])

    # Plotting
    plt.figure(figsize=(14, 7))
    plt.plot(model_pairs, general_proportions, label="General FFN Map", marker='o', linestyle='-')
    plt.plot(model_pairs, entity_proportions, label="Entity FFN Map", marker='s', linestyle='--')
    plt.plot(model_pairs, answer_proportions, label="Relation Answer FFN Map", marker='x', linestyle='-.')
    plt.plot(model_pairs, answer_specific_proportions, label="Answer Specific FFN Map", marker='^', linestyle='-.')

    plt.xticks(rotation=90)
    plt.xlabel("Model Steps")
    plt.ylabel("Switch Proportion")
    plt.title(f"Switch Proportion Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def plot_count_switches(switch_results, relation_name, fact_idx):
    # Prepare data
    model_pairs = list(switch_results.keys())  # Already sorted
    general_counts = []
    entity_counts = []
    answer_counts = []
    answer_specific_counts = []

    # Populate lists for each heatmap type
    for model_pair in model_pairs:
        general_counts.append(switch_results[model_pair]['general_ffn_map']['switch_count'])
        entity_counts.append(switch_results[model_pair]['entity_ffn_map']['switch_count'])
        answer_counts.append(switch_results[model_pair]['relation_answer_ffn_maps']['switch_count'])
        answer_specific_counts.append(switch_results[model_pair]['relation_answer_with_specific']['switch_count'])

    # Plotting
    plt.figure(figsize=(14, 7))
    plt.plot(model_pairs, general_counts, label="General FFN Map", marker='o', linestyle='-')
    plt.plot(model_pairs, entity_counts, label="Entity FFN Map", marker='s', linestyle='--')
    plt.plot(model_pairs, answer_counts, label="Relation Answer FFN Map", marker='x', linestyle='-.')
    plt.plot(model_pairs, answer_specific_counts, label="Answer Specific FFN Map", marker='^', linestyle='-.')

    plt.xticks(rotation=90)
    plt.xlabel("Model Steps")
    plt.ylabel("Switch Counts")
    plt.title(f"Switch Counts Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
THRESHOLD = 0.1
relation_name = "books_written"
fact_idx = 0
overlap = calculate_switches(heatmaps_dict, THRESHOLD, relation_name, fact_idx)
plot_proportion_switches(overlap, relation_name, fact_idx)
plot_count_switches(overlap, relation_name, fact_idx)